# Chapter 10 Companion Notebook: Operational Risk Modeling

This notebook reproduces every worked numerical example from Chapter 10 of *AI in Finance*: the loss distribution approach (a compound Poisson-lognormal Monte Carlo estimate of operational-risk capital), Naive Bayes classification of incident narratives into Basel event types, autoencoder-style anomaly detection for near-misses (with a precision-recall confusion matrix), electronic-communications conduct surveillance, and predictive early warning from key risk indicators via logistic regression.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Combining the four data elements by credibility (Section 10.3.1)

Internal loss data, external consortium data, and scenario analysis disagree.
The Gamma-Poisson conjugate blend resolves them into a single frequency, and
reproduces the `lambda = 12` that the loss distribution approach below uses.

In [1]:
import numpy as np

# Internal loss data: 42 events over 5 years.
cred_n_years, cred_events = 5, 42
cred_lam_internal = cred_events / cred_n_years

# External consortium data plus the scenario workshop, as a Gamma prior whose
# strength beta is measured in years of equivalent experience.
cred_lam_prior, cred_beta = 18.0, 3.0
cred_alpha = cred_lam_prior * cred_beta

cred_Z = cred_n_years / (cred_n_years + cred_beta)
cred_blend = cred_Z * cred_lam_internal + (1 - cred_Z) * cred_lam_prior
cred_post = (cred_alpha + cred_events) / (cred_beta + cred_n_years)
cred_post_sd = np.sqrt(cred_alpha + cred_events) / (cred_beta + cred_n_years)

print(f"internal lambda-hat   = {cred_lam_internal:.2f}")
print(f"prior (ext+scenario)  = {cred_lam_prior:.2f}  held with beta = {cred_beta:.0f} years")
print(f"credibility factor Z  = {cred_Z:.4f}")
print(f"blended lambda        = {cred_blend:.4f}")
print(f"posterior mean        = {cred_post:.4f}   <- same number, read off the posterior")
print(f"posterior sd          = {cred_post_sd:.4f} events/year")


def cred_capital(lam, seed=42):
    """Annual-loss mean and 99.9% quantile, matching Section 10.4's simulation."""
    r = np.random.default_rng(seed)
    c = r.poisson(lam, 500_000)
    a = np.array([r.lognormal(10.5, 1.4, k).sum() if k > 0 else 0.0 for k in c])
    return a.mean(), np.quantile(a, 0.999)


print()
for cred_label, cred_lam in [("internal only", cred_lam_internal),
                             ("credibility blend", cred_post),
                             ("external + scenario only", cred_lam_prior)]:
    cred_el, cred_cap = cred_capital(cred_lam)
    print(f"{cred_label:>26}  lambda={cred_lam:5.2f}  "
          f"EL={cred_el:>11,.0f}  99.9% VaR={cred_cap:>11,.0f}")

cred_el_i, cred_cap_i = cred_capital(cred_lam_internal)
cred_el_b, cred_cap_b = cred_capital(cred_post)
print(f"\ninternal-only understates capital by       {1 - cred_cap_i / cred_cap_b:.1%}")
print(f"internal-only understates expected loss by {1 - cred_el_i / cred_el_b:.1%}")

internal lambda-hat   = 8.40
prior (ext+scenario)  = 18.00  held with beta = 3 years
credibility factor Z  = 0.6250
blended lambda        = 12.0000
posterior mean        = 12.0000   <- same number, read off the posterior
posterior sd          = 1.2247 events/year



             internal only  lambda= 8.40  EL=    812,385  99.9% VaR=  7,188,043


         credibility blend  lambda=12.00  EL=  1,160,689  99.9% VaR=  8,448,026


  external + scenario only  lambda=18.00  EL=  1,740,903  99.9% VaR= 10,255,502



internal-only understates capital by       14.9%
internal-only understates expected loss by 30.0%


## 2. Loss distribution approach: frequency times severity (Section 10.2.3)

In [2]:
import numpy as np
rng = np.random.default_rng(42)
lam, mu_log, sig_log, N = 12.0, 10.5, 1.4, 500_000
print(f"severity median = {np.exp(mu_log):,.0f}; severity mean = {np.exp(mu_log+sig_log**2/2):,.0f}")
counts = rng.poisson(lam, N)
annual = np.array([rng.lognormal(mu_log, sig_log, c).sum() if c>0 else 0.0 for c in counts])
var99, var999 = np.quantile(annual,0.99), np.quantile(annual,0.999)
es99 = annual[annual>=var99].mean()
print(f"expected annual loss = {annual.mean():,.0f}")
print(f"99%   VaR            = {var99:,.0f}")
print(f"99%   ES             = {es99:,.0f}")
print(f"99.9% VaR (capital)  = {var999:,.0f}")
print(f"capital / expected   = {var999/annual.mean():.1f}x")

severity median = 36,316; severity mean = 96,761


expected annual loss = 1,160,689
99%   VaR            = 4,336,930
99%   ES             = 6,114,638
99.9% VaR (capital)  = 8,448,026
capital / expected   = 7.3x


## 3. Incident classification with Naive Bayes (Section 10.3.2)

In [3]:
import math
from collections import Counter
train = {
 "IF": ["trader hid unauthorized position",
        "employee falsified approval to hide loss",
        "unauthorized trade concealed by staff"],
 "SF": ["batch system failed overnight settlement",
        "software deployment caused outage",
        "system outage delayed settlement"]}
vocab=set(); wc={c:Counter() for c in train}; docs={c:len(v) for c,v in train.items()}
for c,ph in train.items():
    for p in ph:
        for w in p.split(): vocab.add(w); wc[c][w]+=1
V=len(vocab); Nt=sum(docs.values()); prior={c:docs[c]/Nt for c in train}
tot={c:sum(wc[c].values()) for c in train}
def score(t):
    return {c: math.log(prior[c]) + sum(math.log((wc[c][w]+1)/(tot[c]+V)) for w in t.split()) for c in train}
sc=score("unauthorized trade hidden by trader")
mx=max(sc.values()); post={c:math.exp(sc[c]-mx) for c in sc}; Z=sum(post.values())
print(f"V = {V}")
for c in sc: print(f"  logP({c}) = {sc[c]:.2f}")
print(f"posterior IF = {post['IF']/Z:.3f}")

V = 24
  logP(IF) = -15.83
  logP(SF) = -18.75
posterior IF = 0.949


## 4. Anomaly detection for near-misses (Section 10.3.3)

In [4]:
from scipy import stats
rng = np.random.default_rng(42)
normal = rng.normal(0,1,(2000,5)); err = (normal**2).sum(1)
thr = np.quantile(err,0.99)
anom = np.array([3.0,2.5,-2.0,3.5,2.0])
print(f"99% threshold = {thr:.1f} (chi-square(5) 99% = {stats.chi2(5).ppf(0.99):.1f})")
print(f"anomalous vector error = {(anom**2).sum():.1f} -> flagged = {(anom**2).sum()>thr}")

rng = np.random.default_rng(11)
tr = rng.normal(0,1,(2000,5)); thr2 = np.quantile((tr**2).sum(1),0.99)
tn = rng.normal(0,1,(500,5)); ta = rng.normal(0,1,(50,5))+1.6
en=(tn**2).sum(1); ea=(ta**2).sum(1)
TP=int((ea>thr2).sum()); FN=50-TP; FP=int((en>thr2).sum()); TN=500-FP
print(f"threshold = {thr2:.1f}; TP={TP} FP={FP} FN={FN} TN={TN}")
print(f"precision = {TP/(TP+FP):.2f}, recall = {TP/(TP+FN):.2f}")

99% threshold = 14.9 (chi-square(5) 99% = 15.1)
anomalous vector error = 35.5 -> flagged = True
threshold = 15.2; TP=28 FP=5 FN=22 TN=495
precision = 0.85, recall = 0.56


## 5. Conduct and rogue-trading surveillance (Section 10.3.4)

In [5]:
train2={
 "SUSPICIOUS":["keep this off the record","do not put this in writing",
               "delete this message after reading","let us not leave a trail"],
 "CLEAN":["please confirm the trade details","the settlement date is tomorrow",
          "can you send the confirmation","the price looks correct to me"]}
vocab=set(); wc={c:Counter() for c in train2}; docs={c:len(v) for c,v in train2.items()}
for c,ph in train2.items():
    for p in ph:
        for w in p.split(): vocab.add(w); wc[c][w]+=1
V=len(vocab); Nt=sum(docs.values()); prior={c:docs[c]/Nt for c in train2}
tot={c:sum(wc[c].values()) for c in train2}
def score2(t):
    return {c: math.log(prior[c]) + sum(math.log((wc[c][w]+1)/(tot[c]+V)) for w in t.split()) for c in train2}
sc=score2("do not leave a trail on this")
mx=max(sc.values()); post={c:math.exp(sc[c]-mx) for c in sc}; Z=sum(post.values())
print(f"V = {V}")
for c in sc: print(f"  logP({c}) = {sc[c]:.2f}")
print(f"posterior SUSPICIOUS = {post['SUSPICIOUS']/Z:.3f}")

V = 36
  logP(SUSPICIOUS) = -23.86
  logP(CLEAN) = -28.99
posterior SUSPICIOUS = 0.994


## 6. Early warning from key risk indicators (Section 10.3.5)

In [6]:
from sklearn.linear_model import LogisticRegression
X = np.array([[12,2,8],[45,9,22],[18,3,10],[50,12,30],
              [8,1,5],[40,8,25],[22,4,12],[55,15,35]], float)
y = np.array([0,1,0,1,0,1,0,1])
Xs = (X - X.mean(0))/X.std(0)
clf = LogisticRegression().fit(Xs,y)
print("standardized coefs (overtime, downtime, turnover):", np.round(clf.coef_[0],2))
for label,row in [("stressed",[48,11,28]),("calm",[10,1,6])]:
    rs=(np.array(row,float)-X.mean(0))/X.std(0)
    print(f"  {label} {row}: P(loss) = {clf.predict_proba(rs.reshape(1,-1))[0,1]:.2f}")

standardized coefs (overtime, downtime, turnover): [0.82 0.69 0.77]
  stressed [48, 11, 28]: P(loss) = 0.90
  calm [10, 1, 6]: P(loss) = 0.07


## 7. Segregation of duties on the entitlement graph (Section 10.10.1)

Permissions are granted to roles, roles nest inside other roles, and a user
inherits the transitive closure. A conflict is therefore a path through the
access graph, not an edge, and a pairwise check on directly-granted
permissions misses most of them.

In [7]:
# Roles that grant permissions directly.
sod_grants = {
    "AP_CLERK": {"create_vendor"},
    "AP_SUPERVISOR": {"approve_payment"},
    "TREASURY_OPS": {"release_payment"},
    "TRADE_ENTRY": {"enter_trade"},
    "TRADE_CONFIRM": {"confirm_trade"},
    "STATIC_DATA": {"modify_static_data"},
    "AP_MANAGER": set(),   # composite: grants nothing itself
    "OPS_ONCALL": set(),   # composite: grants nothing itself
}

# Composite roles nest other roles.
sod_includes = {
    "AP_MANAGER": {"AP_CLERK", "AP_SUPERVISOR"},
    "OPS_ONCALL": {"TREASURY_OPS", "STATIC_DATA"},
}

sod_users = {
    "u1": {"AP_CLERK"},
    "u2": {"AP_SUPERVISOR"},
    "u3": {"AP_CLERK", "AP_SUPERVISOR"},   # conflict visible directly
    "u4": {"AP_MANAGER"},                  # conflict only through nesting
    "u5": {"TRADE_ENTRY"},
    "u6": {"TRADE_ENTRY", "OPS_ONCALL"},
    "u7": {"TRADE_CONFIRM"},
    "u8": {"AP_CLERK", "OPS_ONCALL"},
}

sod_toxic = [
    ("create_vendor", "approve_payment"),
    ("create_vendor", "release_payment"),
    ("enter_trade", "confirm_trade"),
    ("modify_static_data", "release_payment"),
]


def sod_direct(roles):
    """Permissions from a user's own roles, ignoring role nesting."""
    out = set()
    for r in roles:
        out |= sod_grants.get(r, set())
    return out


def sod_effective(roles):
    """Transitive closure: follow nested roles to the permissions they grant."""
    seen, stack, out = set(), list(roles), set()
    while stack:
        r = stack.pop()
        if r in seen:
            continue
        seen.add(r)
        out |= sod_grants.get(r, set())
        stack.extend(sod_includes.get(r, set()))
    return out


def sod_conflicts(perms):
    return [(a, b) for a, b in sod_toxic if a in perms and b in perms]


sod_direct_hits, sod_path_hits = [], []
for sod_user, sod_roles in sod_users.items():
    d = sod_conflicts(sod_direct(sod_roles))
    p = sod_conflicts(sod_effective(sod_roles))
    if d:
        sod_direct_hits.append(sod_user)
    if p:
        sod_path_hits.append(sod_user)
        print(f"{sod_user}: {['+'.join(c) for c in p]}"
              f"  ({'direct' if d else 'via nested role'})")

print(f"\npairwise check on direct permissions finds: {sod_direct_hits} "
      f"({len(sod_direct_hits)} user)")
print(f"path-based check through nested roles finds: {sod_path_hits} "
      f"({len(sod_path_hits)} users)")
print(f"missed by the pairwise check: {sorted(set(sod_path_hits) - set(sod_direct_hits))}")

u3: ['create_vendor+approve_payment']  (direct)
u4: ['create_vendor+approve_payment']  (via nested role)
u6: ['modify_static_data+release_payment']  (via nested role)
u8: ['create_vendor+release_payment', 'modify_static_data+release_payment']  (via nested role)

pairwise check on direct permissions finds: ['u3'] (1 user)
path-based check through nested roles finds: ['u3', 'u4', 'u6', 'u8'] (4 users)
missed by the pairwise check: ['u4', 'u6', 'u8']


## Exercises (match Chapter 10, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [8]:
# Exercise 1: loss distribution approach with lambda=8, Lognormal(11, 1.2), and the sigma=1.6 variant
rng = np.random.default_rng(0)
def lda(lam, mu, sig, M=500_000):
    counts = rng.poisson(lam, M)
    a = np.array([rng.lognormal(mu, sig, c).sum() if c>0 else 0.0 for c in counts])
    return a.mean(), np.quantile(a, 0.999)
for sig in (1.2, 1.6):
    exp, cap = lda(8, 11.0, sig)
    print(f"Exercise 1 (sigma={sig}) -- expected ${exp:,.0f}, 99.9% capital ${cap:,.0f}, ratio {cap/exp:.1f}x")

Exercise 1 (sigma=1.2) -- expected $982,550, 99.9% capital $6,135,355, ratio 6.2x


Exercise 1 (sigma=1.6) -- expected $1,722,952, 99.9% capital $23,244,276, ratio 13.5x
